# APPENDIX B: Gaussian Process Regression (GPR) Model


Supplementary exploratory modeling experiments using Gaussian Process Regression (GPR) were also conducted. While GPR produced competitive predictive performance, particularly for conditional mean estimation, it was not included in the primary comparative analysis in order to keep the main comparison focused on the selected tree-based ensemble methods. Also, compared to the ensemble models included in the main report, GPR was substantially more computationally expesive and more difficult to consistently tune, making the results less stable across repeated validation splits.


## Initialize Environment


In [1]:
# Core Libraries
import math
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# SciPy
from scipy.interpolate import griddata
from scipy.stats import ttest_rel

# Scikit-learn: Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    ConstantKernel as C,
    WhiteKernel
)

# Scikit-learn: Model Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    ShuffleSplit,
    cross_val_score
)

# Scikit-learn: Metrics
from sklearn.metrics import (
    mean_squared_error,
    make_scorer
)

# XGBoost
from xgboost import XGBRegressor


In [2]:
# Global Variables
test_size = 0.20
random_state = 7
n_jobs = 8


## Load & Transform Data


In [3]:
# Load & Transform Training Data
train = pd.read_csv('7406train.csv', header = None)

X = train.iloc[:, 0:2].values
Y = train.iloc[:, 2:].values

mu_hat = Y.mean(axis = 1)
var_hat = Y.var(axis = 1)

print(f"Train  : {train.shape}")
print(f"X Shape: {X.shape}")
print(f"Y Shape: {Y.shape}")

#display(train.head())


Train  : (10000, 202)
X Shape: (10000, 2)
Y Shape: (10000, 200)


In [4]:
# Load Test data
test = pd.read_csv('7406test.csv', header = None)
X_test = test.values

print(f"Test   : {test.shape}")
print(f"X Shape: {X_test.shape}")

#display(test.head())


Test   : (2500, 2)
X Shape: (2500, 2)


In [5]:
# Training & Validation Split
X_train, X_val, mu_train, mu_val, var_train, var_val = train_test_split(
    X, mu_hat, var_hat,
    test_size = test_size,
    random_state = random_state
)

X_train = pd.DataFrame(X_train, columns = ['X1', 'X2'])
X_val = pd.DataFrame(X_val, columns = ['X1', 'X2'])
X_test = pd.DataFrame(X_test, columns = ['X1', 'X2'])


### Gaussian Process Regression (GPR) Model

Purpose:

- Captures smooth nonlinear relationships and interactions using a probabilistic kernel framework
- Flexible nonparametric approach for smooth low-dimensional response surfaces


Model Tuning & Training:

- 7-fold cross-validation used for mean and variance evaluation
- RBF kernel with additive white noise
- Variance modeled on the log scale for stability
- Kernel parameters estimated during training
- Monte Carlo cross-validation used to assess stability


Observations:

- GPR improved mean prediction performance (RMSE = 1.0917)
- Cross-validation and Monte Carlo mean MSE remained consistent near 1.1991
- Variance prediction RMSE = 23.0861 after log-scale variance modeling
- Monte Carlo variance MSE remained near 532.3896
- GPR captured smooth nonlinear structure and interaction effects observed during EDA
- GPR required substantially greater computational cost and more difficult tuning

Conclusion:

- GPR achieved the strongest mean prediction performance among evaluated models
- Validation and resampling results suggested strong generalization with limited overfitting
- Log-scale variance modeling improved stability, although variance estimation remained more difficult
- Kernel-based methods effectively captured smooth nonlinear structure
- GPR remained computationally expensive and more difficult to tune consistently than selected ensemble models


In [6]:
# Cross-Validation
kernel = (
    C(1.0, (1e-3, 1e5)) *
    RBF(length_scale = 0.2, length_scale_bounds = (1e-2, 1e1)) +
    WhiteKernel(noise_level = 1)
)
mse_scorer = make_scorer(mean_squared_error, greater_is_better = False)

# GPR: Mean Model
gpr_mu = GaussianProcessRegressor(
    kernel = kernel,
    alpha = 1e-6,
    normalize_y = True,
    #n_restarts_optimizer = 10,
    random_state = random_state
)

gpr_mean_scores = cross_val_score(
    estimator = gpr_mu,
    X = X_train,
    y = mu_train,
    scoring = mse_scorer,
    cv = 7,
    n_jobs = 1
)

print(f"Mean_MSE: {-gpr_mean_scores.mean():.4f}")
print(f"Mean_MSE_Var: {gpr_mean_scores.var():.4f}")

# GRP: Variance Model
gpr_var = GaussianProcessRegressor(
    kernel = kernel,
    alpha = 1e-6,
    normalize_y = True,
    #n_restarts_optimizer = 10,
    random_state = random_state
)

gpr_var_scores = cross_val_score(
    estimator = gpr_var,
    X = X_train,
    y = np.log(var_train),
    scoring = mse_scorer,
    cv = 7,
    n_jobs = 1
)

print(f"Var_MSE_LogScale: {-gpr_var_scores.mean():.4f}")
print(f"Var_MSE_Var_LogScale: {gpr_var_scores.var():.4f}")


Mean_MSE: 1.1991
Mean_MSE_Var: 0.0023
Var_MSE_LogScale: 0.0127
Var_MSE_Var_LogScale: 0.0000


In [7]:
# GPR: Final Mean Model
gpr_mu.fit(X_train, mu_train)

mu_pred_val_gpr = gpr_mu.predict(X_val)

mse_mu = mean_squared_error(mu_val, mu_pred_val_gpr)
rmse_mu = np.sqrt(mse_mu)

print(f"GPR Mean MSE: {mse_mu:.4f}")
print(f"GPR Mean RMSE: {rmse_mu:.4f}")

# GPR: Final Variance Model
gpr_var.fit(X_train, np.log(var_train))

log_var_pred_val_gpr = gpr_var.predict(X_val)
var_pred_val_gpr = np.exp(log_var_pred_val_gpr)

mse_var = mean_squared_error(var_val, var_pred_val_gpr)
rmse_var = np.sqrt(mse_var)

print(f"GPR Variance MSE: {mse_var:.4f}")
print(f"GPR Variance RMSE: {rmse_var:.4f}")


GPR Mean MSE: 1.1917
GPR Mean RMSE: 1.0917
GPR Variance MSE: 532.9674
GPR Variance RMSE: 23.0861


In [8]:
# Monte Carlo Cross-Validation
cv = ShuffleSplit(n_splits = 100, test_size = test_size, random_state = random_state)

rows = []
for i, (train_idx, val_idx) in enumerate(cv.split(X_train)):

    # Data Split
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    mu_tr = mu_train[train_idx]
    mu_val = mu_train[val_idx]

    var_tr = var_train[train_idx]
    var_val = var_train[val_idx]

    log_var_tr = np.log(var_tr)

    # Fit GPR Models
    gpr_mu.fit(X_tr, mu_tr)
    gpr_var.fit(X_tr, log_var_tr)

    # Predictions
    mu_gpr = gpr_mu.predict(X_val)

    log_var_gpr = gpr_var.predict(X_val)
    var_gpr = np.exp(log_var_gpr)

    # MSE Results
    rows.append({
        'Mean_MSE': mean_squared_error(mu_val, mu_gpr),
        'Var_MSE': mean_squared_error(var_val, var_gpr)
    })

# Results Table
results_df = pd.DataFrame(rows)

summary_df = pd.DataFrame({
    'Mean_MSE': [results_df['Mean_MSE'].mean()],
    'Var_MSE': [results_df['Var_MSE'].mean()],
    'Mean_MSE_Var': [results_df['Mean_MSE'].var()],
    'Var_MSE_Var': [results_df['Var_MSE'].var()]
}).round(4)

display(summary_df)


/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/anaconda3/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


,Mean_MSE,Var_MSE,Mean_MSE_Var,Var_MSE_Var
0,1.1992,532.3896,0.0021,323.9916
